# Checkpoint FIAP - RPA & NLP (Cross-Tab Data Fusion)

**Objetivo:** Este projeto realiza a extração automatizada multi-domínio da plataforma World Monitor e cruza blocos de dados tabulares caóticos para criar um Contexto Sintético ("Prompt_Ready") útil em pipelines de RAG (Retrieval-Augmented Generation).

---
## Integrantes do Grupo
* **Nicolas Lemos Ribeiro**: RM 553273
* **Ricardo de Paiva Melo**: RM 565522
* **Luís Fernando de Oliveira Salgado**: RM 561401
* **Pedro Leal Murad**: RM 565460
* **Murilo Benhossi**: RM 562358
---

### Etapa 1: Instalação das Dependências (Setup)
Abaixo instalamos o **Playwright** (motor headless realístico de Chromium que não depende de WebDrivers físicos como o Selenium), o **nest_asyncio** (fundamental para rodar o pool assíncrono do Playwright dentro do loop nativo do Google Colab) e bibliotecas de apresentação visual como TQDM para logs.

In [ ]:
!pip install nest_asyncio playwright pandas tqdm
!playwright install-deps chromium
!playwright install chromium

In [ ]:
import nest_asyncio
import asyncio
import pandas as pd
import io
import os
import json
from datetime import datetime
from playwright.async_api import async_playwright
from tqdm.notebook import tqdm
from IPython.display import display

# Sem o nest_asyncio, o motor do Colab conflitaria com o event_loop do Playwright
nest_asyncio.apply()

### Etapa 2: Robótica Web (RPA com Playwright)
Aqui inicia nossa automação avançada. Em vez de raspar o HTML de forma frágil, nosso robô adota uma estratégia de **Espera Ativa (`wait_for_selector`)**. Como o World Monitor é um Single Page Application (SPA), apenas clicar em "baixar CSV" resultaria em um arquivo vazio se a API ainda não preencheu o front-end. 

**Descoberta técnica importante:** Ao analisar os JSONs exportados, identificamos que o botão de Export do World Monitor **não inclui as notícias** nos domínios World, Tech e Finance (retorna `news: []` vazio), embora elas estejam visíveis nos painéis da página. Isso ocorre porque as notícias são carregadas por uma API de "digest" separada (19+ categorias de fontes) que popula os painéis renderizados, mas não é capturada pelo mecanismo de export nativo.

**Solução — Dupla Estratégia de Extração:**
1. **DOM Scraping:** Após aguardar o carregamento completo de todos os painéis, um script JavaScript percorre todos os links da página e identifica manchetes de notícias externas (filtrando navegação interna, Polymarket, etc.), extraindo título, fonte e timestamp de cada uma.
2. **JSON Export:** O download nativo do JSON é mantido para capturar dados de **Markets** (índices financeiros) e **Predictions** (mercados preditivos Polymarket), além das news que já venham preenchidas (Commodity e Good News).
3. **Merge inteligente:** As notícias do DOM são injetadas no array de `news` do JSON, com deduplicação por título, resultando em um dataset unificado e completo por domínio.

In [ ]:
async def extrair_multi_dominio():
    # Coleta sequencial ampliada com DUPLA ESTRATÉGIA de extração:
    # 1) DOM Scraping: extrai manchetes diretamente dos painéis renderizados na SPA
    # 2) JSON Export:  baixa o arquivo nativo (markets, predictions, e news quando disponível)
    #
    # Motivo: o Export JSON do World Monitor retorna news:[] vazio para world/tech/finance,
    # embora as notícias estejam visíveis nos painéis da página (carregadas via digest API separada).
    # A combinação das duas estratégias garante cobertura completa de todos os 5 eixos temáticos.

    dominios = {
        'world': 'https://www.worldmonitor.app/',
        'tech': 'https://tech.worldmonitor.app/',
        'finance': 'https://finance.worldmonitor.app/',
        'commodity': 'https://commodity.worldmonitor.app/',
        'happy': 'https://happy.worldmonitor.app/'
    }
    
    os.makedirs('raw_data', exist_ok=True)
    
    # JavaScript Scraper: percorre todos os links da página e identifica manchetes de notícias
    # nos painéis renderizados (Alimentação Intel, IA/ML, Governo, etc.)
    # Estrutura DOM de cada notícia: [fonte] → [link com manchete] → [timestamp]
    JS_DOM_SCRAPER = '''() => {
        const results = [];
        const seen = new Set();
        const allLinks = document.querySelectorAll('a[href]');
        
        for (const link of allLinks) {
            const href = link.href;
            const title = link.textContent.trim();
            
            // Filtra: apenas links externos com texto de manchete (> 20 caracteres)
            if (!href || title.length < 20) continue;
            if (href.includes('worldmonitor.app') || href.includes('polymarket.com') || href.includes('github.com')) continue;
            if (href.includes('protomaps.com') || href.includes('openstreetmap.org')) continue;
            if (href.startsWith('#') || href.startsWith('javascript:')) continue;
            
            // Deduplica por título
            if (seen.has(title)) continue;
            seen.add(title);
            
            // Localiza o nome da fonte (elemento irmão anterior ao link no DOM)
            let source = 'World Monitor';
            const prevEl = link.previousElementSibling;
            if (prevEl) {
                let text = prevEl.textContent.trim()
                    .replace(/ALERT/g, '').replace(/●/g, '').trim();
                if (text.length > 1 && text.length < 80) source = text;
            }
            
            // Localiza o timestamp (elemento irmão posterior ao link)
            let lastUpdated = '';
            const nextEl = link.nextElementSibling;
            if (nextEl) {
                lastUpdated = nextEl.textContent.trim().replace(/文$/g, '').trim();
            }
            
            results.push({
                primaryTitle: title,
                primaryLink: href,
                primarySource: source,
                lastUpdated: lastUpdated
            });
        }
        return results;
    }'''
    
    async with async_playwright() as p:
        # Configurando o Anti-Bot e Gravador Visual Acadêmico
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            record_video_dir="videos/"
        )
        
        for nome, url in tqdm(dominios.items(), desc="🌍 Extraindo Domínios"):
            page = await context.new_page()
            try:
                print(f"\n[RPA] Acessando {url}...")
                # networkidle: aguarda a SPA terminar TODOS os redirects internos e chamadas de API
                # (o World Monitor redireciona de / para /?lat=...&zoom=...&layers=... ao carregar)
                await page.goto(url, wait_until='networkidle', timeout=60000)
                
                # Espera Ativa: aguarda a UI estabilizar após o redirect do SPA
                print(f"[RPA] Aguardando a UI carregar e dados popularem via API (Espera robótica)...")
                try:
                    await page.wait_for_selector('button.export-btn', state='attached', timeout=20000)
                except:
                    # Fallback: se o seletor do export não existir, aguarda qualquer botão
                    print(f"   [WAIT] export-btn não encontrado, usando fallback...")
                    await page.wait_for_selector('button', state='attached', timeout=10000)
                
                # Cold-sleep ampliado para garantir que TODOS os painéis de notícias 
                # terminem de carregar (digest API carrega 19+ categorias assincronamente)
                await asyncio.sleep(10)
                # Garante que a rede estabilizou antes de ler o DOM
                await page.wait_for_load_state('networkidle')
                
                # ═══ ESTRATÉGIA 1: Scraping do DOM (captura TODAS as manchetes dos painéis) ═══
                dom_news = []
                for tentativa in range(3):
                    try:
                        print(f"[RPA] Extraindo notícias dos painéis renderizados no DOM...")
                        dom_news = await page.evaluate(JS_DOM_SCRAPER)
                        print(f"   [DOM] {len(dom_news)} manchetes extraídas dos painéis")
                        break
                    except Exception as e:
                        if tentativa < 2:
                            print(f"   [DOM] Tentativa {tentativa+1} falhou ({e}), aguardando estabilização...")
                            await asyncio.sleep(5)
                            # Aguarda qualquer navegação pendente finalizar
                            try:
                                await page.wait_for_load_state('networkidle')
                            except:
                                pass
                        else:
                            print(f"   [DOM] ⚠️ Todas as tentativas falharam: {e}")
                
                # ═══ ESTRATÉGIA 2: Export JSON nativo (markets + predictions + news quando disponível) ═══
                json_data = {'news': [], 'markets': [], 'predictions': [], 'timestamp': None}
                try:
                    print(f"[RPA] Forçando clique no botão de exportação (Ignorando Overlays/Animações)...")
                    await page.evaluate('''() => {
                        const btn = document.querySelector('button.export-btn') || document.querySelector('.export-panel-container button');
                        if (btn) btn.click();
                    }''')
                    await asyncio.sleep(1)  # Aguarda o menu dropdown expandir
                    
                    async with page.expect_download(timeout=10000) as download_info:
                        await page.evaluate('''() => {
                            const options = document.querySelectorAll('button.export-option[data-format="json"]');
                            const btn = Array.from(options).find(e => e.offsetParent !== null) || options[0];
                            if (btn) btn.click();
                        }''')
                    download = await download_info.value
                    
                    # Lê o JSON exportado para memória
                    temp_path = f"raw_data/_temp_{nome}.json"
                    await download.save_as(temp_path)
                    with open(temp_path, 'r', encoding='utf-8') as f:
                        json_data = json.load(f)
                    os.remove(temp_path)
                    
                    n_news = len(json_data.get('news', []))
                    n_mkt = len(json_data.get('markets', []))
                    n_pred = len(json_data.get('predictions', []))
                    print(f"   [JSON Export] {n_news} news, {n_mkt} markets, {n_pred} predictions")
                except Exception as e:
                    print(f"   [JSON Export] ⚠️ Falhou (usando apenas DOM): {e}")
                
                # ═══ MERGE: injeta notícias do DOM no array de news do JSON ═══
                # Evita duplicatas comparando por título
                existing_titles = {
                    n.get('primaryTitle', n.get('title', ''))
                    for n in json_data.get('news', [])
                }
                added = 0
                for item in dom_news:
                    if item['primaryTitle'] not in existing_titles:
                        json_data['news'].append(item)
                        existing_titles.add(item['primaryTitle'])
                        added += 1
                
                if added > 0:
                    print(f"   [MERGE] +{added} notícias do DOM incorporadas ao dataset")
                
                # Salva o JSON enriquecido (DOM + Export combinados)
                caminho_arquivo = f"raw_data/{nome}_raw.json"
                with open(caminho_arquivo, 'w', encoding='utf-8') as f:
                    json.dump(json_data, f, ensure_ascii=False, indent=2)
                
                total = len(json_data.get('news', []))
                print(f"✅ {nome}: {total} notícias totais salvas → {caminho_arquivo}")
                
            except Exception as e:
                print(f"❌ Erro na extração de {nome}: {e}")
            finally:
                await page.close()
                
        await context.close()
        await browser.close()
        print("\n✅ Fase 1 [RPA] Finalizada com Dupla Extração (DOM + JSON)!")

# Dispara no event_loop do Jupyter
await extrair_multi_dominio()

### Etapa 3: Tratamento de Dados (Data Engineering)
O World Monitor exporta dados em JSON estruturado com três chaves principais: `news` (notícias agregadas), `markets` (índices de mercado) e `predictions` (mercados preditivos do Polymarket).

Graças à **Dupla Estratégia de Extração** (DOM Scraping + JSON Export), o array de `news` agora contém manchetes de **todos os 5 domínios**: as provenientes do export nativo (Commodity e Good News) e as extraídas diretamente dos painéis renderizados (World, Tech e Finance).

Para garantir representatividade máxima na tabela final, também **normalizamos as predictions para o mesmo schema das news** e as incorporamos ao dataset unificado, marcando cada registro com seu `Tipo_Conteudo` (News ou Prediction).

**Nota sobre timestamps:** As notícias extraídas via DOM possuem timestamps relativos ("ontem", "há 7 horas") que são preservados como texto — o `pd.to_datetime(errors='coerce')` os converte para `NaT`, priorizando a integridade dos dados reais.

In [ ]:
def parse_json_data(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"⚠️ {file_path} não encontrado!")
        return None, None, None

    # Extração direta e elegante do Schema JSON (agora incluindo predictions)
    df_news = pd.DataFrame(data.get('news', []))
    df_markets = pd.DataFrame(data.get('markets', []))
    df_predictions = pd.DataFrame(data.get('predictions', []))
    
    return df_news, df_markets, df_predictions

In [ ]:
print("Executando Data Parsing da Ingestão JSON...")
news_world, mkt_world, pred_world = parse_json_data('raw_data/world_raw.json')
news_tech, mkt_tech, pred_tech = parse_json_data('raw_data/tech_raw.json')
news_fin, mkt_fin, pred_fin = parse_json_data('raw_data/finance_raw.json')
news_com, mkt_com, pred_com = parse_json_data('raw_data/commodity_raw.json')
news_happy, mkt_happy, pred_happy = parse_json_data('raw_data/happy_raw.json')

# --- Normalização: transforma predictions no mesmo schema das news ---
# Predictions vêm com: title, yesPrice, volume, url, endDate, tags
# Normalizamos para: primaryTitle, primarySource, primaryLink, lastUpdated
def normalizar_predictions(df_pred):
    if df_pred is None or df_pred.empty:
        return pd.DataFrame()
    df = pd.DataFrame()
    df['primaryTitle'] = df_pred['title'].apply(
        lambda t: f"[Previsão] {t}" 
    )
    # Enriquece o título com a probabilidade e volume quando disponíveis
    if 'yesPrice' in df_pred.columns:
        df['primaryTitle'] = df_pred.apply(
            lambda r: f"[Previsão | Prob: {r.get('yesPrice', 'N/A')}% | Vol: ${r.get('volume', 0):,.0f}] {r['title']}", 
            axis=1
        )
    df['primarySource'] = 'Polymarket'
    df['primaryLink'] = df_pred.get('url', '')
    df['lastUpdated'] = df_pred.get('endDate', pd.NaT)
    df['Tipo_Conteudo'] = 'Prediction'
    return df

pred_world_norm = normalizar_predictions(pred_world)
pred_tech_norm = normalizar_predictions(pred_tech)
pred_fin_norm = normalizar_predictions(pred_fin)
pred_com_norm = normalizar_predictions(pred_com)
pred_happy_norm = normalizar_predictions(pred_happy)

# --- Concatenação de NEWS ---
dfs_news = []
for df, dominio in [
    (news_world, 'World'), 
    (news_tech, 'Tech'), 
    (news_fin, 'Finance'),
    (news_com, 'Commodity'),
    (news_happy, 'Good News')
]:
    if df is not None and not df.empty:
        df = df.copy()
        df['Dominio_Origem'] = dominio
        df['Tipo_Conteudo'] = 'News'
        dfs_news.append(df)

# --- Concatenação de PREDICTIONS normalizadas ---
for df, dominio in [
    (pred_world_norm, 'World'),
    (pred_tech_norm, 'Tech'),
    (pred_fin_norm, 'Finance'),
    (pred_com_norm, 'Commodity'),
    (pred_happy_norm, 'Good News')
]:
    if df is not None and not df.empty:
        df = df.copy()
        df['Dominio_Origem'] = dominio
        dfs_news.append(df)

if dfs_news:
    df_global_news = pd.concat(dfs_news, ignore_index=True)
    
    tit_col = 'primaryTitle' if 'primaryTitle' in df_global_news.columns else 'title'
    lnk_col = 'primaryLink' if 'primaryLink' in df_global_news.columns else 'link'
    
    if tit_col in df_global_news.columns and lnk_col in df_global_news.columns:
        df_global_news.drop_duplicates(subset=[tit_col, lnk_col], inplace=True)
    
    # Sanitização de Datas (ISO 8601)
    pub_col = 'lastUpdated' if 'lastUpdated' in df_global_news.columns else 'pubDate'
    if pub_col in df_global_news.columns:
        df_global_news['Published'] = pd.to_datetime(df_global_news[pub_col], errors='coerce')
        df_global_news.sort_values(by='Published', ascending=False, inplace=True)
else:
    df_global_news = pd.DataFrame()

# Capturamos TODOS os mercados para uma visão global mais rica (não só finance)
dfs_markets = []
for df, dominio in [
    (mkt_world, 'World'), (mkt_tech, 'Tech'), (mkt_fin, 'Finance'),
    (mkt_com, 'Commodity'), (mkt_happy, 'Good News')
]:
    if df is not None and not df.empty:
        dfs_markets.append(df)

if dfs_markets:
    df_global_markets = pd.concat(dfs_markets, ignore_index=True)
    sym_col = 'symbol' if 'symbol' in df_global_markets.columns else 'Symbol'
    df_global_markets.drop_duplicates(subset=[sym_col], inplace=True)
else:
    df_global_markets = pd.DataFrame()

# Diagnóstico rápido
print(f"\n📊 Conteúdo consolidado:")
for dom in ['World', 'Tech', 'Finance', 'Commodity', 'Good News']:
    n = len(df_global_news[df_global_news['Dominio_Origem'] == dom]) if not df_global_news.empty else 0
    print(f"   {dom}: {n} registros")
print(f"   Total: {len(df_global_news)} registros | Markets: {len(df_global_markets)} indicadores")


### Etapa 4: Cross-Tab Data Fusion (O Cérebro da Operação para a NLP/RAG)

É aqui que o "ouro" é formado para a equipe de AI (Etapa 4 do Padrão Lima na prova). 
Jogar uma "Notícia" isolada dentro de uma RAG gera embeddings contextuais matematicamente genéricos. O que fazemos aqui é o **Enriquecimento Estrutural por Metadados Sintéticos**.

Pegamos a planilha separada de **Índices Macroeconômicos** extraída do RPA e mesclamos em linguagem natural com a aba de **Notícias/Geopolítica**, gerando a coluna `Prompt_Ready` (a notícia + o sentimento e saúde da bolsa mundial atrelados juntos ao arquivo `nlp_ready_fusion.csv`).

In [ ]:
# Preparando o Contexto de "Temperatura" Global via Bolsa/Commodities
cenario_macro = "Estável"
if not df_global_markets.empty:
    indicadores = []
    sym_col = 'symbol' if 'symbol' in df_global_markets.columns else 'Symbol'
    chg_col = 'change' if 'change' in df_global_markets.columns else 'Change'
    
    for _, row in df_global_markets.head(5).iterrows(): 
        sym = row.get(sym_col, 'N/A')
        chg = str(row.get(chg_col, '0')).strip()
        indicadores.append(f"{sym} ({chg})")
    
    texto_indices = " | ".join(indicadores)
    cenario_macro = f"Mercado Misto. Indicadores Chaves simultâneos: {texto_indices}."

def construir_prompt_ready(row):
    tit_col = 'primaryTitle' if 'primaryTitle' in row else 'title'
    src_col = 'primarySource' if 'primarySource' in row else 'source'
    
    titulo = row.get(tit_col, 'Sem Título')
    fonte = row.get(src_col, row.get('Source', 'Desconhecida'))
    data = row.get('Published')
    dominio = row.get('Dominio_Origem', 'Global')
    tipo = row.get('Tipo_Conteudo', 'News')
    
    data_str = data.isoformat() if pd.notnull(data) else 'Data Recente'
    
    # Diferencia o template entre News e Predictions para melhor semântica NLP
    if tipo == 'Prediction':
        contexto = (
            f"Aba Taxonômica da Plataforma: [{dominio}]. "
            f"Previsão de mercado (Polymarket): '{titulo}'. "
            f"Data de encerramento da aposta: {data_str}. "
            f"Para baseamento analítico, o Contexto Macroeconômico/Bolsa Mundial era: {cenario_macro}"
        )
    else:
        contexto = (
            f"Aba Taxonômica da Plataforma: [{dominio}]. "
            f"Em {data_str}, a fonte '{fonte}' relatou essa manchete central: "
            f"'{titulo}'. "
            f"Para baseamento analítico da resposta, repara-se que o Contexto Macroeconômico/Bolsa Mundial neste mesmo ponto do tempo era: {cenario_macro}"
        )
    return contexto

if not df_global_news.empty:
    df_global_news['Contexto_Sintetico'] = cenario_macro
    df_global_news['Prompt_Ready'] = df_global_news.apply(construir_prompt_ready, axis=1)

    tit_col = 'primaryTitle' if 'primaryTitle' in df_global_news.columns else 'title'
    src_col = 'primarySource' if 'primarySource' in df_global_news.columns else 'source'
    lnk_col = 'primaryLink' if 'primaryLink' in df_global_news.columns else 'link'

    cols_ordem = ['Published', 'Dominio_Origem', 'Tipo_Conteudo', src_col, tit_col, lnk_col, 'Contexto_Sintetico', 'Prompt_Ready']
    df_final = df_global_news[[c for c in cols_ordem if c in df_global_news.columns]]
    
    df_final.to_csv('nlp_ready_fusion.csv', index=False)
    
    print("\n✅ Mágica de Dados concluída. Arquivo final pronto para ingestão do Bert/GPT (RAG)!")
    print(f"📊 {len(df_final)} registros totais | News: {len(df_final[df_final['Tipo_Conteudo']=='News'])} | Predictions: {len(df_final[df_final['Tipo_Conteudo']=='Prediction'])}")
    print(f"🌍 Domínios representados: {df_final['Dominio_Origem'].unique().tolist()}")
    print("\n💎 Amostra do Enriquecimento (Coluna Prompt_Ready pronta para Ingestão):\n")
    
    pd.set_option('display.max_colwidth', 150)
    display(df_final[['Dominio_Origem', 'Tipo_Conteudo', tit_col, 'Prompt_Ready']].head(10))
else:
    print("Dataset de cruzamento final resultou vazio.")
